<a href="https://colab.research.google.com/github/NagaTejaSriCore/Buliding-a-Game-Using-Crew/blob/main/Buliding__game_crew_sec3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U crewai

In [2]:
!pip install "crewai[google-genai]"

In [3]:
!pip install "crewai[tools]"

In [4]:
from crewai import Agent,LLM,Task,Crew,Process
from google.colab import userdata

In [5]:
from crewai_tools import SerperDevTool

In [6]:
serper_tool=SerperDevTool(api_key=userdata.get("serper_api_key"))

#**Creative Game Designer Agent**
```
   role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",
   
   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
```

In [7]:
from crewai import Agent,LLM
from google.colab import userdata

In [20]:
llm=LLM(
    model="gemini/gemini-1.5-flash",
    api_key=userdata.get("google_api")
)

In [9]:
designer=Agent(
    role="Creative Game Designer",

   goal="Come up with fun, feasible game concepts and detailed mechanics based on user idea",

   backstory=
     "You are an experienced game designer."
     "You excel at turning vague ideas into clear, exciting game designs including:"
     "- core loop, rules, win/lose conditions"
     "- basic entities (player, enemies, items)"
     "- controls and feel"
     "Keep it simple enough to implement in pure Python + Pygame in one file.",
    llm=llm,
    verbose=True
)

#**Senior Python Game Developer Agent**

```
role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
```

In [10]:
developer=Agent(
    role="Senior Python Game Developer",

goal="Write clean, working Python code (using Pygame) for the described game",

backstory=
       "You are a senior software engineer specialized in Python game development with Pygame."
       "You write structured, readable code with:"
       "- Proper game loop, event handling, drawing"
       "- Comments explaining key parts"
       "- Error handling where needed"
       "You always produce a complete, runnable .py file.",
    llm=llm,
    verbose=True,
    tools=[serper_tool]
)

#**QA Engineer & Code Reviewer Agent**

    role="QA Engineer & Code Reviewer",

    goal="Test, review, and improve the code for bugs, playability, and completeness",
    
    backstory=
        "You are a meticulous QA engineer and code reviewer."
        "You carefully check:"
        "- Does the code run without errors?"
        "- Does it implement ALL the designed features?"
        "- Is it fun/playable? Any obvious balance issues?"
        "- Code style, variable names, comments"
        "Suggest fixes or small improvements and output the FINAL improved code.",


In [11]:

QA_engineer=Agent(role="QA Engineer & Code Reviewer",

goal="Test, review, and improve the code for bugs, playability, and completeness",

backstory=
    "You are a meticulous QA engineer and code reviewer."
    "You carefully check:"
    "- Does the code run without errors?"
    "- Does it implement ALL the designed features?"
    "- Is it fun/playable? Any obvious balance issues?"
    "- Code style, variable names, comments"
    "Suggest fixes or small improvements and output the FINAL improved code.",
                  llm=llm,
                  verbose=True)

### **Game Designing Task**

    description=
        "Take the user's game idea: {game_idea}"
        "1. Clarify and expand it into a fun, simple 2D game"
        "2. Describe: objective, controls, entities, win/lose"
        "3. Keep scope small (one level, basic mechanics)"
        "Output format:"
        "## Game Design Document"
        "- Title: ..."
        "- Genre: ..."
        "- Objective: ..."
        "- Controls: ..."
        "- Entities: ..."
        "- Mechanics: ...",

In [12]:
task_design=Task(
    description=
    "Take the user's game idea: {game_idea}"
    "1. Clarify and expand it into a fun, simple 2D game"
    "2. Describe: objective, controls, entities, win/lose"
    "3. Keep scope small (one level, basic mechanics)"
    "Output format:"
    "## Game Design Document"
    "- Title: ..."
    "- Genre: ..."
    "- Objective: ..."
    "- Controls: ..."
    "- Entities: ..."
    "- Mechanics: ...",
    expected_output="A clear Markdown Game design Document",
    agent=designer
)

### **Coding Task**

    description=
        "Using the game design from the previous task"
        "Write a COMPLETE, standalone Python script using Pygame that implements the game."
        "- Include import pygame, sys, random (if needed)"
        "- Full game loop, init, events, update, draw"
        "- Make it runnable with python game.py"
        "- Add simple comments"
        "- The main game loop must be exposed in the python code, it should not be inside any function like main",
        "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",

In [13]:
task_code=Task(
    description=
    "Using the game design from the previous task"
    "Write a COMPLETE, standalone Python script using Pygame that implements the game."
    "- Include import pygame, sys, random (if needed)"
    "- Full game loop, init, events, update, draw"
    "- Make it runnable with python game.py"
    "- Add simple comments"
    "- The main game loop must be exposed in the python code, it should not be inside any function like main"
    "- Final answer MUST be ONLY the Python code and Instructions on how to play the game",
    expected_output="A Complete runnable Pygame Python script",
    agent=developer,
    context=[task_design]
)

###**Review Task**

    description=
        "Review the Python code from the previous task."
        "1. Check for syntax/runtime errors"
        "2. Verify it matches the design document"
        "3. Test mentally: does it have init, loop, quit handling, drawing?"
        "4. Suggest fixes/improvements if needed"
        "5. Output the FINAL, improved, ready-to-run code"
        "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    

In [14]:
task_review=Task(
    description=
    "Review the Python code from the previous task."
    "1. Check for syntax/runtime errors"
    "2. Verify it matches the design document"
    "3. Test mentally: does it have init, loop, quit handling, drawing?"
    "4. Suggest fixes/improvements if needed"
    "5. Output the FINAL, improved, ready-to-run code"
    "Your final answer MUST be ONLY the complete Python code along with the instructions on how to play the game",
    expected_output="Final polished, runnable Pygame Python script and instructions on how to play the game",
    agent=QA_engineer,
    context=[task_design,task_code]
)

In [15]:
game_crew=Crew(
    agents=[designer,developer,QA_engineer],
    tasks=[task_design,task_code,task_review],
    process=Process.sequential,
    verbose=True
)

In [18]:
game_idea="A fun endless runner where a character jumps over obstacles"
result = game_crew.kickoff(inputs={"game_idea": game_idea})
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 80e66deb-08c9-4940-a268-6e03c823d7a9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│  ID: 891339cf-d995-40ad-ab3a-601dcb981b80                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Task: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Game Design Document                                                                                        │
│                                                                                                                 │
│  -   **Title:** Pixel Leaper                                                                                    │
│  -   **Genre:** 2D Endless Runner, Platformer                                                                   │
│  -   **Objective:** Survive for as long as possible by jumping over incoming obstacles, achieving the highest   │
│  possible score.                                                                                                │
│  -   **Controls:**                                                                                              │
│      *   **Spacebar:** Jump. The player character automatically moves forward (the world scrolls around them).  │
│  Pressing Spacebar will make the character jump upwards. Holding it down does not affect jump height, it's a    │
│  single impulse.                                                                                                │
│  -   **Entities:**                                                                                              │
│      *   **Player Character (The Leaper):**                                                                     │
│          *   A simple rectangular sprite (or small animated character).                                         │
│          *   Automatically 'runs' to the right (visually, the ground and obstacles move left).                  │
│          *   Can jump, subject to gravity.                                                                      │
│          *   Fixed horizontally on screen (e.g., at 20% of screen width), only its vertical position changes    │
│  with jumps and gravity.                                                                                        │
│      *   **Ground:**                                                                                            │
│          *   A continuous, scrolling horizontal line or strip at the bottom of the screen.                      │
│          *   Provides a surface for the player to run on and land after jumps.                                  │
│      *   **Obstacles:**                                                                                         │
│          *   Simple rectangular blocks of varying heights and widths.                                           │
│          *   Examples: Single short block, single tall block, two blocks with a small gap, etc.                 │
│          *   Spawn off-screen to the right and move left towards the player.                                    │
│          *   Collision with player results in Game Over.                                                        │
│      *   **Score Display:**                                                                                     │
│          *   Text rendered on-screen, showing the current score.                                                │
│          *   Updates continuously.                                                                              │
│  -   **Mechanics:**                                                                                             │
│      *   **Continuous Movement:** The game world (groun

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Take the user's game idea: A fun endless runner where a character jumps over obstacles1. Clarify and     │
│  expand it into a fun, simple 2D game2. Describe: objective, controls, entities, win/lose3. Keep scope small    │
│  (one level, basic mechanics)Output format:## Game Design Document- Title: ...- Genre: ...- Objective: ...-     │
│  Controls: ...- Entities: ...- Mechanics: ...                                                                   │
│  Agent: Creative Game Designer                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Using the game design from the previous taskWrite a COMPLETE, standalone Python script using Pygame      │
│  that implements the game.- Include import pygame, sys, random (if needed)- Full game loop, init, events,       │
│  update, draw- Make it runnable with python game.py- Add simple comments- The main game loop must be exposed    │
│  in the python code, it should not be inside any function like main- Final answer MUST be ONLY the Python code  │
│  and Instructions on how to play the game                                                                       │
│  ID: 0bed10d3-b1b3-4997-913b-bf8f98130501                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (8 outputs left out)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Task: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  import pygame                                                                                                  │
│  import sys                                                                                                     │
│  import random                                                                                                  │
│                                                                                                                 │
│  # Initialize Pygame                                                                                            │
│  pygame.init()                                                                                                  │
│                                                                                                                 │
│  # Screen dimensions                                                                                            │
│  SCREEN_WIDTH = 800                                                                                             │
│  SCREEN_HEIGHT = 400                                                                                            │
│                                                                                                                 │
│  # Colors                                                                                                       │
│  WHITE = (255, 255, 255)                                                                                        │
│  BLACK = (0, 0, 0)                                                                                              │
│  RED = (255, 0, 0)                                                                                              │
│  BLUE = (0, 0, 255)                                                                                             │
│  GREY = (100, 100, 100)                                                                                         │
│                                                                                                                 │
│  # Game constants                                                                                               │
│  GROUND_HEIGHT = 50                                                                                             │
│  PLAYER_WIDTH = 20                                                                                              │
│  PLAYER_HEIGHT = 40                                                                                             │
│  PLAYER_X = 80  # Fixed horizontal position of the player                                                       │
│                                                                                                                 │
│  GRAVITY = 0.8                                                                                                  │
│  JUMP_STRENGTH = -15 # Negative for upward movement (y-axis is inverted)                                        │
│                                                                                                                 │
│  OBSTACLE_MIN_WIDTH = 20                                                                                        │
│  OBSTACLE_MAX_WIDTH = 50                               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the Python code from the previous task.1. Check for syntax/runtime errors2. Verify it matches     │
│  the design document3. Test mentally: does it have init, loop, quit handling, drawing?4. Suggest                │
│  fixes/improvements if needed5. Output the FINAL, improved, ready-to-run codeYour final answer MUST be ONLY     │
│  the complete Python code along with the instructions on how to play the game                                   │
│  Agent: QA Engineer & Code Reviewer                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

```python
import pygame
import sys
import random

# Initialize Pygame
pygame.init()

# Screen dimensions
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 400

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
BLUE = (0, 0, 255)
GREY = (100, 100, 100)

# Game constants
GROUND_HEIGHT = 50
PLAYER_WIDTH = 20
PLAYER_HEIGHT = 40
PLAYER_X = 80  # Fixed horizontal position of the player

GRAVITY = 0.8
JUMP_STRENGTH = -15 # Negative for upward movement (y-axis is inverted)

OBSTACLE_MIN_WIDTH = 20
OBSTACLE_MAX_WIDTH = 50
OBSTACLE_MIN_HEIGHT = 30
OBSTACLE_MAX_HEIGHT = 80
OBSTACLE_START_X = SCREEN_WIDTH + 50 # Obstacles start off-screen to the right

INITIAL_GAME_SPEED = 5
GAME_SPEED_INCREMENT_PER_FRAME = 0.0005 # Speed increases slightly over time each frame
OBSTACLE_SPAWN_INTERVAL = 1500 # Initial interval in milliseconds
MIN_OBSTACLE_SPAWN_INTERVAL = 700 # Minimum interval for obstacle spawning
SPAWN_INTERVAL_DECREMENT_PER_100_SCORE = 50 # How much spawn interval decreases per 100 score p

In [30]:
%%writefile main.py

import pygame
import random

# Initialize Pygame
pygame.init()

# Screen dimensions
SCREEN_WIDTH = 800
SCREEN_HEIGHT = 400

# Colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (255, 0, 0)
BLUE = (0, 0, 255)
GREY = (100, 100, 100)

# Game constants
GROUND_HEIGHT = 50
PLAYER_WIDTH = 20
PLAYER_HEIGHT = 40
PLAYER_X = 80

GRAVITY = 0.8
JUMP_STRENGTH = -15

OBSTACLE_MIN_WIDTH = 20
OBSTACLE_MAX_WIDTH = 50
OBSTACLE_MIN_HEIGHT = 30
OBSTACLE_MAX_HEIGHT = 80
OBSTACLE_START_X = SCREEN_WIDTH + 50

INITIAL_GAME_SPEED = 5
GAME_SPEED_INCREMENT = 0.005

OBSTACLE_SPAWN_INTERVAL = 1500
MIN_OBSTACLE_SPAWN_INTERVAL = 700

SCORE_INCREMENT_PER_FRAME = 0.1

# Set up display
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Pixel Leaper")

# Clock
clock = pygame.time.Clock()

# Fonts
font = pygame.font.Font(None, 36)
game_over_font = pygame.font.Font(None, 64)


# Obstacle class
class Obstacle:
    def __init__(self, x, y, width, height, speed):
        self.rect = pygame.Rect(x, y, width, height)
        self.speed = speed

    def update(self):
        self.rect.x -= self.speed

    def draw(self, screen):
        pygame.draw.rect(screen, BLUE, self.rect)


# Reset game function
def reset_game():
    global player_y
    global player_y_velocity
    global is_grounded
    global obstacles
    global score
    global game_active
    global game_speed
    global last_obstacle_spawn_time

    player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
    player_y_velocity = 0
    is_grounded = True
    obstacles = []
    score = 0
    game_active = True
    game_speed = INITIAL_GAME_SPEED
    last_obstacle_spawn_time = pygame.time.get_ticks()


# Initial game state
player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
player_y_velocity = 0
is_grounded = True

obstacles = []

score = 0

game_active = True

game_speed = INITIAL_GAME_SPEED

last_obstacle_spawn_time = pygame.time.get_ticks()


# Main game loop
running = True

while running:

    # Events
    for event in pygame.event.get():

        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.KEYDOWN:

            if event.key == pygame.K_SPACE:

                if game_active:

                    if is_grounded:
                        player_y_velocity = JUMP_STRENGTH
                        is_grounded = False

                else:
                    reset_game()

    # Game updates
    if game_active:

        # Gravity
        player_y_velocity += GRAVITY
        player_y += player_y_velocity

        # Ground collision
        if player_y >= SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT:
            player_y = SCREEN_HEIGHT - GROUND_HEIGHT - PLAYER_HEIGHT
            player_y_velocity = 0
            is_grounded = True

        # Increase difficulty
        game_speed += GAME_SPEED_INCREMENT
        score += SCORE_INCREMENT_PER_FRAME

        # Spawn obstacles
        current_time = pygame.time.get_ticks()

        current_spawn_interval = max(
            MIN_OBSTACLE_SPAWN_INTERVAL,
            OBSTACLE_SPAWN_INTERVAL - (int(score) // 100) * 50
        )

        if current_time - last_obstacle_spawn_time > current_spawn_interval:

            obstacle_width = random.randint(
                OBSTACLE_MIN_WIDTH,
                OBSTACLE_MAX_WIDTH
            )

            obstacle_height = random.randint(
                OBSTACLE_MIN_HEIGHT,
                OBSTACLE_MAX_HEIGHT
            )

            obstacle_y = (
                SCREEN_HEIGHT
                - GROUND_HEIGHT
                - obstacle_height
            )

            new_obstacle = Obstacle(
                OBSTACLE_START_X,
                obstacle_y,
                obstacle_width,
                obstacle_height,
                game_speed
            )

            obstacles.append(new_obstacle)

            last_obstacle_spawn_time = current_time

        # Update obstacles
        for obstacle in list(obstacles):

            obstacle.update()

            if obstacle.rect.right < 0:
                obstacles.remove(obstacle)

        # Collision detection
        player_rect = pygame.Rect(
            PLAYER_X,
            player_y,
            PLAYER_WIDTH,
            PLAYER_HEIGHT
        )

        for obstacle in obstacles:

            if player_rect.colliderect(obstacle.rect):
                game_active = False

        # Draw background
        screen.fill(WHITE)

        # Draw ground
        pygame.draw.rect(
            screen,
            GREY,
            (0, SCREEN_HEIGHT - GROUND_HEIGHT,
             SCREEN_WIDTH, GROUND_HEIGHT)
        )

        # Draw player
        pygame.draw.rect(screen, RED, player_rect)

        # Draw obstacles
        for obstacle in obstacles:
            obstacle.draw(screen)

        # Draw score
        score_text = font.render(
            f"Score: {int(score)}",
            True,
            BLACK
        )

        screen.blit(score_text, (10, 10))

    else:

        # Game Over screen
        screen.fill(WHITE)

        game_over_text = game_over_font.render(
            "GAME OVER",
            True,
            BLACK
        )

        game_over_rect = game_over_text.get_rect(
            center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 - 30)
        )

        screen.blit(game_over_text, game_over_rect)

        final_score_text = font.render(
            f"Final Score: {int(score)}",
            True,
            BLACK
        )

        final_score_rect = final_score_text.get_rect(
            center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 20)
        )

        screen.blit(final_score_text, final_score_rect)

        restart_text = font.render(
            "Press SPACE to Restart",
            True,
            BLACK
        )

        restart_rect = restart_text.get_rect(
            center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 60)
        )

        screen.blit(restart_text, restart_rect)

    # Update display
    pygame.display.flip()

    # FPS
    clock.tick(60)

pygame.quit()

Overwriting main.py


# **Do not edit any of the below code**

In [22]:
!pip install pygbag pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 4.1 MB/s eta 0:00:00


# **How to get pyngrok API Key?**
- Go to https://dashboard.ngrok.com/get-started/your-authtoken
- Copy your authtoken
- Run this cell with YOUR token:

In [31]:
ngrok_auth_token=userdata.get("NGROK_AUTH_TOKEN")

In [32]:
!ngrok authtoken "36H00JrYLbZGYFsnTKbXayKhA6W_5i51Tr2zaaTUBF8cabsfa"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [33]:
import subprocess
import time
from pyngrok import ngrok

# Step 1: Build the game
print("🔨 Building game...")
build_result = subprocess.run(
    ['pygbag', '--build', '--version', '0.9', '--PYBUILD', '3.12', '--cdn', 'https://pygame-web.github.io/archives/0.9/', 'main.py'],
    capture_output=True, text=True
)
print(build_result.stdout)

if build_result.returncode != 0:
    print("❌ Build failed:")
    print(build_result.stderr)
else:
    print("✅ Build successful!")

    # Step 2: Start HTTP server
    print("\n🚀 Starting server...")
    server = subprocess.Popen(
        ['python', '-m', 'http.server', '8000', '--directory', '/content/build/web'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

    # Wait for server to start
    time.sleep(5)

    # Step 3: Create ngrok tunnel
    try:
        print("🌐 Creating public URL...")
        public_url = ngrok.connect(8000)

        print("\n" + "="*60)
        print("🎮 YOUR GAME IS READY!")
        print("="*60)
        print(f"\n🔗 Click here to play: {public_url}")
        print("\n📝 How to play:")
        print("   • Press SPACE or Click to jump")
        print("="*60)

    except Exception as e:
        print(f"\n❌ Error creating tunnel: {e}")
        print("\n💡 Make sure you ran Cell 3 with a valid ngrok token")

🔨 Building game...

Serving python files from [/content/build/web]

with no security/performance in mind, i'm just a test tool : don't rely on me


SUMMARY
________________________

# the app folder
app_folder=/content

# artefacts directory
build_dir=/content/build/web

# cache directory
cache=/content/build/web-cache

# the window title and icon name
app_name=content

# package name, better make it unique
package=web.pygame.content-1778160443

# icons:  96x96 for desktop, 16x16 for web
icon=favicon.png

# js/wasm provider
cdn=https://pygame-web.github.io/archives/0.9/

now packing application ....


Ignored dirs: []
Ignored files: []
Now in /
     /main.py
Now in /sample_data
     /sample_data/README.md
     /sample_data/anscombe.json
     /sample_data/california_housing_test.csv
     /sample_data/mnist_test.csv
     /sample_data/california_housing_train.csv
     /sample_data/mnist_train_small.csv
optimizing /content
	/content : main.py
	/content : sample_data/README.md
	/content : s